In [2]:
from bs4 import BeautifulSoup as Soup
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores.chroma import Chroma
from langchain_community.document_loaders.recursive_url_loader import RecursiveUrlLoader
from langchain_core.documents import Document
from langchain_community.embeddings.ollama import OllamaEmbeddings
import chromadb

import time
import json
import os


In [ ]:
URL: str = "https://skcet.ac.in/"


In [ ]:
text_splitter = RecursiveCharacterTextSplitter()


In [ ]:
loader = RecursiveUrlLoader(
    url=URL,
    max_depth=2,
    extractor=lambda x: Soup(x, "html5lib").text.strip().replace("\n", " "),
    check_response_status=True,
    continue_on_failure=True,
    base_url=URL,
)
docs = loader.load_and_split(text_splitter=text_splitter)


In [ ]:
len(docs)


In [ ]:
os.makedirs("./data", exist_ok=True)
with open("./data/skcet.json", "w") as f:
    json.dump(list(map(lambda doc: doc.dict(), docs)), f, indent=4)


In [4]:
# create docs from "./data/skcet.jsonl"
docs: list[Document] = list(
    map(
        lambda doc: Document(
            page_content=doc["page_content"],
            metadata=doc["metadata"],
        ),
        json.load(open("./data/skcet.json")),
    )
)


In [ ]:
def add_documents_to_chroma(
    documents: list[Document],
    store: Chroma,
    batch_size: int = 10,
    sleep_time: float = 1.0,
    start: int = 0,
) -> list[str]:
    """
    add_documents adds documents to the store while preventing rate limits.

    Parameters
    ----------
    documents : list[Document]
        The documents to be added to the store
    store : Chroma
        The store to add the documents to
    batch_size : int, optional
        The number of documents to add at once, by default 10
    sleep_time : float | Literal["auto"], optional
        The time to sleep between requests, by default "auto".

    Returns
    -------
    list[str]
        The ids of the documents added to the store
    """
    ids: list[str] = []
    for i in range(start, len(documents), batch_size):
        try:
            ids.extend(store.add_documents(documents[i : i + batch_size]))
            print(f"Added documents {i} to {i + batch_size}")
            time.sleep(sleep_time)
        except Exception as e:
            print(e, f"retrying {i} to {i + batch_size}")
            time.sleep(5)
            ids.extend(
                add_documents_to_chroma(documents, store, batch_size, sleep_time, i)
            )
    return ids


In [ ]:
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large", skip_empty=True)
embeddings = OllamaEmbeddings(
    model="llama3",
)
new_client = chromadb.PersistentClient()
chroma = Chroma(
    client=new_client,
    collection_name="skcet",
    embedding_function=embeddings,
)

chroma.add_documents(docs)
# add_documents_to_chroma(docs, chroma)
